<a href="https://colab.research.google.com/github/radhikatyagi388/Ai_60_Day_Challange/blob/main/Debugging_AI_Systems_Systematically.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 🚀 Day 33 — AI System Debugging with LangSmith

## 🎯 Focus Area

### AI System Debugging

AI system bugs are different from traditional software bugs.

A traditional software system usually fails by producing an error or exception. An AI system can execute successfully but still produce an incorrect, irrelevant, or ungrounded answer.

Therefore, debugging an AI system requires investigating the complete execution path:

**User Query → Retrieval → Context → Prompt → Generation → Final Answer**

This project builds a systematic debugging workflow around real failures identified in the Day 29 evaluation.

---

# 🧠 Project Objective

The objective of this milestone is to build an observable and reproducible debugging workflow for an AI/RAG system.

The workflow will:

1. Enable full LangSmith execution tracing.
2. Reproduce three real failures from the Day 29 evaluation.
3. Inspect the complete execution trace for each failure.
4. Isolate individual pipeline components.
5. Identify the exact point where the error enters the system.
6. Implement structured JSON logging.
7. Fix two of the three failures.
8. Re-run the evaluation suite.
9. Compare before-and-after evaluation scores.
10. Verify that improvements do not introduce regressions.
11. Create a reusable AI debugging runbook.

---

# 🛠️ Tools & Technologies

- Python
- Google Colab
- LangChain
- LangSmith
- OpenAI API
- FAISS / Vector Store
- Python `logging`
- JSON
- Evaluation dataset from Day 29

---

# 🏗️ Debugging Architecture

The AI pipeline will be investigated as three independent components:

```text
                    USER QUERY
                        │
                        ▼
              ┌──────────────────┐
              │    RETRIEVAL     │
              │                  │
              │ Vector Search     │
              │ Top-K Documents   │
              └────────┬─────────┘
                       │
                       ▼
              ┌──────────────────┐
              │ PROMPT           │
              │ CONSTRUCTION     │
              │                  │
              │ Context + Query  │
              └────────┬─────────┘
                       │
                       ▼
              ┌──────────────────┐
              │   GENERATION     │
              │                  │
              │      LLM         │
              └────────┬─────────┘
                       │
                       ▼
                 FINAL ANSWER

In [3]:
# ================================================================
# DAY 33 — AI SYSTEM DEBUGGING
# COMPLETE GOOGLE COLAB CODE
#
# NO OPENAI API
#
# Technologies:
#   Python
#   LangChain
#   LangSmith (optional)
#   Hugging Face
#   Sentence Transformers
#   FAISS
#   FLAN-T5
#   Structured JSON logging
# ================================================================


# ================================================================
# 1. INSTALL DEPENDENCIES
# ================================================================

!pip -q install -U \
    langchain \
    langchain-community \
    langchain-huggingface \
    langsmith \
    faiss-cpu \
    sentence-transformers \
    transformers \
    accelerate


# ================================================================
# 2. IMPORTS
# ================================================================

import os
import json
import time
import uuid
import logging
import warnings
import torch

from datetime import datetime, timezone

warnings.filterwarnings("ignore")

from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate

from langchain_huggingface import HuggingFaceEmbeddings

from langchain_community.vectorstores import FAISS

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM
)

print("✅ All libraries imported")


# ================================================================
# 3. LANGSMITH CONFIGURATION
# ================================================================
#
# IMPORTANT:
# No OPENAI_API_KEY is required.
#
# LangSmith is optional.
#
# To enable LangSmith:
#
# Colab → Secrets → Add:
#
# LANGSMITH_API_KEY
#
# ================================================================

LANGSMITH_ENABLED = False

try:

    from google.colab import userdata

    try:

        LANGSMITH_API_KEY = userdata.get(
            "LANGSMITH_API_KEY"
        )

        if LANGSMITH_API_KEY:

            os.environ[
                "LANGSMITH_API_KEY"
            ] = LANGSMITH_API_KEY

            os.environ[
                "LANGSMITH_TRACING"
            ] = "true"

            os.environ[
                "LANGCHAIN_TRACING_V2"
            ] = "true"

            os.environ[
                "LANGCHAIN_PROJECT"
            ] = "day30-ai-debugging"

            LANGSMITH_ENABLED = True

            print("✅ LangSmith tracing enabled")

        else:

            print(
                "⚠️ No LangSmith API key found"
            )

            print(
                "Running without dashboard tracing"
            )

    except Exception:

        print(
            "⚠️ LANGSMITH_API_KEY not found"
        )

        print(
            "Running locally"
        )

except Exception:

    print(
        "⚠️ Colab Secrets unavailable"
    )


print(
    "OpenAI API:",
    "NOT USED"
)


# ================================================================
# 4. STRUCTURED JSON LOGGING
# ================================================================

class JSONFormatter(
    logging.Formatter
):

    def format(self, record):

        log_data = {

            "timestamp":
                datetime.now(
                    timezone.utc
                ).isoformat(),

            "session_id":
                getattr(
                    record,
                    "session_id",
                    None
                ),

            "step_name":
                getattr(
                    record,
                    "step_name",
                    record.name
                ),

            "input_summary":
                getattr(
                    record,
                    "input_summary",
                    None
                ),

            "output_summary":
                getattr(
                    record,
                    "output_summary",
                    None
                ),

            "latency_ms":
                getattr(
                    record,
                    "latency_ms",
                    None
                )
        }

        return json.dumps(
            log_data,
            ensure_ascii=False
        )


logger = logging.getLogger(
    "day30_debugger"
)

logger.setLevel(
    logging.INFO
)

logger.handlers.clear()


console_handler = logging.StreamHandler()

console_handler.setFormatter(
    JSONFormatter()
)

logger.addHandler(
    console_handler
)


file_handler = logging.FileHandler(
    "debugging_logs.jsonl"
)

file_handler.setFormatter(
    JSONFormatter()
)

logger.addHandler(
    file_handler
)


def log_step(
    session_id,
    step_name,
    input_summary,
    output_summary,
    latency_ms
):

    logger.info(

        "",

        extra={

            "session_id":
                session_id,

            "step_name":
                step_name,

            "input_summary":
                str(
                    input_summary
                )[:1500],

            "output_summary":
                str(
                    output_summary
                )[:1500],

            "latency_ms":
                round(
                    latency_ms,
                    2
                )
        }
    )


print(
    "✅ Structured JSON logging enabled"
)

print(
    "📁 debugging_logs.jsonl created"
)


# ================================================================
# 5. CREATE KNOWLEDGE BASE
# ================================================================
#
# This is a self-contained demonstration dataset.
#
# For your actual Day 30 submission, replace these documents
# with your Day 29 documents/vector database.
# ================================================================

documents = [

    Document(

        page_content="""
        LangChain is a framework for developing applications
        powered by language models. It provides components for
        prompts, retrieval, document processing, agents and chains.
        """,

        metadata={
            "source": "langchain"
        }

    ),

    Document(

        page_content="""
        Retrieval-Augmented Generation, commonly called RAG,
        combines information retrieval with language generation.
        A retriever searches a knowledge base for relevant documents,
        and the language model uses those documents as context.
        """,

        metadata={
            "source": "rag"
        }

    ),

    Document(

        page_content="""
        LangSmith is a platform for observing, evaluating and
        debugging applications built with language models.
        It provides traces that allow developers to inspect
        individual execution steps.
        """,

        metadata={
            "source": "langsmith"
        }

    ),

    Document(

        page_content="""
        Chunk size and chunk overlap are important retrieval
        configuration parameters. Larger chunks provide more
        surrounding context, while overlap helps preserve information
        that crosses chunk boundaries.
        """,

        metadata={
            "source": "chunking"
        }

    ),

    Document(

        page_content="""
        Structured logging makes AI debugging easier because each
        execution can record timestamps, session identifiers,
        component names, input summaries, output summaries and
        execution latency.
        """,

        metadata={
            "source": "logging"
        }

    ),

    Document(

        page_content="""
        Grounded generation means that an AI model should produce
        answers supported by the retrieved context rather than
        inventing unsupported facts.
        """,

        metadata={
            "source": "grounding"
        }

    )

]


print(
    f"✅ Knowledge base: {len(documents)} documents"
)


# ================================================================
# 6. EMBEDDING MODEL
# ================================================================

print(
    "\nLoading embedding model..."
)

embedding_model = HuggingFaceEmbeddings(

    model_name=
        "sentence-transformers/all-MiniLM-L6-v2"

)

print(
    "✅ Embedding model loaded"
)


# ================================================================
# 7. FAISS VECTOR STORE
# ================================================================

vectorstore = FAISS.from_documents(

    documents,

    embedding_model

)

print(
    "✅ FAISS vector store created"
)


# ================================================================
# 8. RETRIEVER
# ================================================================

retriever = vectorstore.as_retriever(

    search_kwargs={
        "k": 3
    }

)

print(
    "✅ Retriever created"
)


# ================================================================
# 9. LOAD LOCAL FLAN-T5
# ================================================================
#
# IMPORTANT:
#
# We DO NOT use:
#
# pipeline("text2text-generation")
#
# because current Colab Transformers versions may not expose
# that pipeline task.
#
# Instead we directly call model.generate().
# ================================================================

print(
    "\nLoading local FLAN-T5-small..."
)

MODEL_NAME = "google/flan-t5-small"


tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)


model = AutoModelForSeq2SeqLM.from_pretrained(

    MODEL_NAME,

    tie_word_embeddings=False

)


device = torch.device(

    "cuda"
    if torch.cuda.is_available()
    else "cpu"

)


model = model.to(
    device
)

model.eval()


print(
    "✅ FLAN-T5 loaded"
)

print(
    "Device:",
    device
)

print(
    "OpenAI API: NOT USED"
)


# ================================================================
# 10. LOCAL GENERATION FUNCTION
# ================================================================

def generate_answer(
    prompt
):

    inputs = tokenizer(

        prompt,

        return_tensors="pt",

        truncation=True,

        max_length=512

    )


    inputs = {

        key:
            value.to(device)

        for key, value
        in inputs.items()

    }


    with torch.no_grad():

        outputs = model.generate(

            **inputs,

            max_new_tokens=150,

            do_sample=False

        )


    answer = tokenizer.decode(

        outputs[0],

        skip_special_tokens=True

    )


    return answer.strip()


# ================================================================
# 11. TEST LOCAL MODEL
# ================================================================

test_prompt = """
Answer the question using only the context.

Context:
LangSmith is a platform for observing, evaluating and
debugging applications built with language models.

Question:
What is LangSmith used for?

Answer:
"""


test_answer = generate_answer(
    test_prompt
)


print("\n")
print("=" * 70)
print("LOCAL MODEL TEST")
print("=" * 70)

print(
    test_answer
)

print(
    "=" * 70
)


# ================================================================
# 12. ORIGINAL PROMPT
# ================================================================

ORIGINAL_PROMPT = """
Answer the question using the context.

Context:
{context}

Question:
{question}

Answer:
"""


original_prompt = PromptTemplate(

    template=ORIGINAL_PROMPT,

    input_variables=[
        "context",
        "question"
    ]

)


# ================================================================
# 13. IMPROVED PROMPT
# ================================================================

IMPROVED_PROMPT = """
You are a factual and grounded question-answering assistant.

Use ONLY the information explicitly provided in the context.

Rules:

1. Do not use outside knowledge.
2. Do not invent facts.
3. Do not make unsupported claims.
4. If the answer is not contained in the context,
   respond exactly:
   I don't have enough information in the context.
5. Keep the answer concise.
6. Every factual statement must be supported by the context.

Context:
{context}

Question:
{question}

Answer:
"""


improved_prompt = PromptTemplate(

    template=IMPROVED_PROMPT,

    input_variables=[
        "context",
        "question"
    ]

)


print(
    "✅ Original prompt created"
)

print(
    "✅ Improved prompt created"
)


# ================================================================
# 14. RETRIEVAL ISOLATION
# ================================================================

def run_retrieval(

    query,

    session_id

):

    start = time.perf_counter()


    docs = retriever.invoke(
        query
    )


    latency = (

        time.perf_counter()
        - start

    ) * 1000


    output = []


    for i, doc in enumerate(docs):

        output.append({

            "rank":
                i + 1,

            "content":
                doc.page_content,

            "metadata":
                doc.metadata

        })


    log_step(

        session_id,

        "retrieval",

        query,

        output,

        latency

    )


    return docs


# ================================================================
# 15. PROMPT CONSTRUCTION ISOLATION
# ================================================================

def build_prompt(

    question,

    docs,

    session_id,

    prompt_template

):

    context = "\n\n".join(

        f"Document {i + 1}:\n"
        f"{doc.page_content}"

        for i, doc
        in enumerate(docs)

    )


    start = time.perf_counter()


    final_prompt = prompt_template.format(

        context=context,

        question=question

    )


    latency = (

        time.perf_counter()
        - start

    ) * 1000


    log_step(

        session_id,

        "prompt_construction",

        {

            "question":
                question,

            "documents":
                len(docs)

        },

        final_prompt,

        latency

    )


    return final_prompt


# ================================================================
# 16. GENERATION ISOLATION
# ================================================================

def run_generation(

    final_prompt,

    session_id,

    step_name="generation"

):

    start = time.perf_counter()


    answer = generate_answer(
        final_prompt
    )


    latency = (

        time.perf_counter()
        - start

    ) * 1000


    log_step(

        session_id,

        step_name,

        final_prompt,

        answer,

        latency

    )


    return answer


# ================================================================
# 17. COMPLETE PIPELINE
# ================================================================

def debug_pipeline(

    question,

    prompt_template=original_prompt

):

    session_id = str(
        uuid.uuid4()
    )


    print("\n")
    print("=" * 100)

    print(
        "AI DEBUGGING SESSION"
    )

    print("=" * 100)


    print(
        "Session ID:",
        session_id
    )


    print(
        "Question:",
        question
    )


    # ------------------------------------------------------------
    # RETRIEVAL
    # ------------------------------------------------------------

    docs = run_retrieval(

        question,

        session_id

    )


    print("\n")
    print("-" * 100)

    print(
        "STEP 1 — RETRIEVAL"
    )

    print("-" * 100)


    for i, doc in enumerate(docs):

        print(
            f"\nDOCUMENT {i + 1}"
        )

        print(
            doc.page_content.strip()
        )

        print(
            "Metadata:",
            doc.metadata
        )


    # ------------------------------------------------------------
    # PROMPT
    # ------------------------------------------------------------

    final_prompt = build_prompt(

        question,

        docs,

        session_id,

        prompt_template

    )


    print("\n")
    print("-" * 100)

    print(
        "STEP 2 — PROMPT CONSTRUCTION"
    )

    print("-" * 100)


    print(
        final_prompt
    )


    # ------------------------------------------------------------
    # GENERATION
    # ------------------------------------------------------------

    answer = run_generation(

        final_prompt,

        session_id

    )


    print("\n")
    print("-" * 100)

    print(
        "STEP 3 — GENERATION"
    )

    print("-" * 100)


    print(
        answer
    )


    print("\n")
    print("=" * 100)


    return {

        "session_id":
            session_id,

        "question":
            question,

        "documents":
            docs,

        "prompt":
            final_prompt,

        "answer":
            answer

    }


# ================================================================
# 18. THREE REPRODUCIBLE FAILURE CASES
# ================================================================
#
# IMPORTANT:
#
# Replace these with your ACTUAL Day 29 failed queries if available.
# ================================================================

FAILURES = [

    {

        "id":
            "F1",

        "query":
            "What is LangSmith used for?",

        "expected":
            "LangSmith is used for observing, evaluating and debugging applications built with language models.",

        "correctness_before":
            1,

        "groundedness_before":
            1

    },

    {

        "id":
            "F2",

        "query":
            "What is retrieval augmented generation?",

        "expected":
            "RAG combines information retrieval with language generation.",

        "correctness_before":
            1,

        "groundedness_before":
            1

    },

    {

        "id":
            "F3",

        "query":
            "Why is chunk overlap useful?",

        "expected":
            "Chunk overlap helps preserve information that crosses chunk boundaries.",

        "correctness_before":
            1,

        "groundedness_before":
            1

    }

]


# ================================================================
# 19. REPRODUCE THREE FAILURES
# ================================================================

failure_results = {}


for failure in FAILURES:

    print("\n\n")
    print("#" * 100)

    print(
        f"REPRODUCING {failure['id']}"
    )

    print("#" * 100)


    result = debug_pipeline(

        failure["query"],

        original_prompt

    )


    failure_results[
        failure["id"]
    ] = result


# ================================================================
# 20. FAILURE ANALYSIS
# ================================================================

def analyze_failure(

    failure,

    result

):

    print("\n")
    print("=" * 90)

    print(
        f"FAILURE ANALYSIS — {failure['id']}"
    )

    print("=" * 90)


    print(
        "\nQuery:"
    )

    print(
        failure["query"]
    )


    print(
        "\nExpected:"
    )

    print(
        failure["expected"]
    )


    print(
        "\nActual:"
    )

    print(
        result["answer"]
    )


    print(
        "\nRetrieved evidence:"
    )


    for i, doc in enumerate(
        result["documents"]
    ):

        print(
            f"\nDocument {i + 1}:"
        )

        print(
            doc.page_content.strip()
        )


    print("\n")
    print(
        "ROOT CAUSE INVESTIGATION"
    )

    print("-" * 60)

    print("""
Check the following:

[1] RETRIEVAL
    Is the required information present
    in the retrieved documents?

[2] PROMPT
    Was the retrieved information correctly
    passed into the prompt?

[3] GENERATION
    Did the model produce an answer supported
    by the provided context?

Possible root cause:

    Retrieval Failure
    Prompt Failure
    Generation Failure
""")


for failure in FAILURES:

    analyze_failure(

        failure,

        failure_results[
            failure["id"]
        ]

    )


# ================================================================
# 21. IMPROVED PIPELINE
# ================================================================

def improved_pipeline(

    question

):

    session_id = str(
        uuid.uuid4()
    )


    print("\n")
    print("=" * 100)

    print(
        "IMPROVED PIPELINE"
    )

    print("=" * 100)


    print(
        "Session ID:",
        session_id
    )


    # ------------------------------------------------------------
    # RETRIEVAL
    # ------------------------------------------------------------

    docs = run_retrieval(

        question,

        session_id

    )


    # ------------------------------------------------------------
    # IMPROVED CONTEXT
    # ------------------------------------------------------------

    context = "\n\n".join(

        f"Document {i + 1}:\n"
        f"{doc.page_content}"

        for i, doc
        in enumerate(docs)

    )


    # ------------------------------------------------------------
    # IMPROVED PROMPT
    # ------------------------------------------------------------

    start = time.perf_counter()


    final_prompt = improved_prompt.format(

        context=context,

        question=question

    )


    latency = (

        time.perf_counter()
        - start

    ) * 1000


    log_step(

        session_id,

        "improved_prompt",

        question,

        final_prompt,

        latency

    )


    # ------------------------------------------------------------
    # IMPROVED GENERATION
    # ------------------------------------------------------------

    answer = run_generation(

        final_prompt,

        session_id,

        step_name=
            "improved_generation"

    )


    print("\n")
    print(
        "FINAL ANSWER:"
    )

    print(
        answer
    )


    return {

        "session_id":
            session_id,

        "question":
            question,

        "documents":
            docs,

        "prompt":
            final_prompt,

        "answer":
            answer

    }


# ================================================================
# 22. APPLY TWO FIXES
# ================================================================
#
# Fix 1:
# Improve prompt grounding instructions.
#
# Fix 2:
# Increase retrieval k from 3 to 4 for a second retriever.
# ================================================================

print("\n")
print("=" * 100)

print(
    "CREATING FIX #2 — IMPROVED RETRIEVAL"
)

print("=" * 100)


improved_retriever = vectorstore.as_retriever(

    search_kwargs={
        "k": 4
    }

)


def improved_retrieval_pipeline(

    question

):

    session_id = str(
        uuid.uuid4()
    )


    start = time.perf_counter()


    docs = improved_retriever.invoke(
        question
    )


    latency = (

        time.perf_counter()
        - start

    ) * 1000


    log_step(

        session_id,

        "fixed_retrieval",

        question,

        [
            doc.page_content[:500]
            for doc in docs
        ],

        latency

    )


    context = "\n\n".join(

        f"Document {i + 1}:\n"
        f"{doc.page_content}"

        for i, doc
        in enumerate(docs)

    )


    final_prompt = improved_prompt.format(

        context=context,

        question=question

    )


    answer = run_generation(

        final_prompt,

        session_id,

        step_name=
            "fixed_generation"

    )


    return {

        "session_id":
            session_id,

        "answer":
            answer,

        "documents":
            docs,

        "prompt":
            final_prompt

    }


# ================================================================
# 23. RUN FIXES ON FIRST TWO FAILURES
# ================================================================

fixed_results = {}


for failure in FAILURES[:2]:

    print("\n\n")
    print("#" * 100)

    print(
        f"TESTING FIX — {failure['id']}"
    )

    print("#" * 100)


    fixed_result = improved_retrieval_pipeline(

        failure["query"]

    )


    fixed_results[
        failure["id"]
    ] = fixed_result


    print("\nExpected:")

    print(
        failure["expected"]
    )


    print("\nNew answer:")

    print(
        fixed_result["answer"]
    )


# ================================================================
# 24. EVALUATION HELPER
# ================================================================
#
# Since this notebook avoids external LLM APIs, the final score
# evaluator is implemented as a transparent keyword/evidence
# check.
#
# For your real Day 29 project, replace this with your actual
# evaluation function.
# ================================================================

def simple_evaluate(

    answer,

    expected

):

    answer_lower = answer.lower()

    expected_lower = expected.lower()


    # Extract important words
    words = [

        word.strip(
            ".,!?;:"
        )

        for word in expected_lower.split()

        if len(word) > 3

    ]


    if not words:

        return 0


    matches = sum(

        word in answer_lower

        for word in words

    )


    ratio = matches / len(words)


    if ratio >= 0.75:

        return 3

    elif ratio >= 0.45:

        return 2

    elif ratio >= 0.20:

        return 1

    else:

        return 0


# ================================================================
# 25. BEFORE / AFTER EVALUATION
# ================================================================

comparison = []


for failure in FAILURES:

    fid = failure["id"]


    original_answer = failure_results[
        fid
    ]["answer"]


    before_score = simple_evaluate(

        original_answer,

        failure["expected"]

    )


    if fid in fixed_results:

        after_answer = fixed_results[
            fid
        ]["answer"]


        after_score = simple_evaluate(

            after_answer,

            failure["expected"]

        )


    else:

        after_answer = original_answer

        after_score = before_score


    comparison.append({

        "failure":
            fid,

        "query":
            failure["query"],

        "before_score":
            before_score,

        "after_score":
            after_score,

        "improved":
            after_score > before_score,

        "before_answer":
            original_answer,

        "after_answer":
            after_answer

    })


# ================================================================
# 26. DISPLAY EVALUATION
# ================================================================

print("\n")
print("=" * 100)

print(
    "BEFORE vs AFTER EVALUATION"
)

print("=" * 100)


for row in comparison:

    print("\n")
    print(
        row["failure"]
    )

    print(
        "Before:",
        row["before_score"]
    )

    print(
        "After:",
        row["after_score"]
    )

    print(
        "Improved:",
        row["improved"]
    )


# ================================================================
# 27. SAVE EVALUATION RESULTS
# ================================================================

with open(

    "before_after_evaluation.json",

    "w"

) as file:

    json.dump(

        comparison,

        file,

        indent=4,

        ensure_ascii=False

    )


print(
    "\n✅ before_after_evaluation.json saved"
)


# ================================================================
# 28. SAVE DEBUGGING REPORT
# ================================================================

debugging_report = {

    "project":
        "Day 30 — AI System Debugging",

    "openai_api_used":
        False,

    "langsmith_enabled":
        LANGSMITH_ENABLED,

    "generation_model":
        MODEL_NAME,

    "embedding_model":
        "sentence-transformers/all-MiniLM-L6-v2",

    "vector_store":
        "FAISS",

    "timestamp":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "failures":
        FAILURES,

    "evaluation":
        comparison

}


with open(

    "day30_debugging_report.json",

    "w"

) as file:

    json.dump(

        debugging_report,

        file,

        indent=4,

        ensure_ascii=False

    )


print(
    "✅ day30_debugging_report.json saved"
)


# ================================================================
# 29. CREATE DEBUGGING RUNBOOK
# ================================================================

runbook = """
# AI SYSTEM DEBUGGING RUNBOOK

## 1. Reproduce the Failure

Run the exact failing query again.

Record:

- Query
- Expected answer
- Actual answer
- Evaluation score
- Session ID

## 2. Inspect LangSmith Trace

Open the trace for the failed session.

Follow:

Query
→ Retrieval
→ Prompt
→ Generation
→ Final Answer

## 3. Check Retrieval

Determine whether the correct evidence was retrieved.

Investigate:

- Chunk size
- Chunk overlap
- Top-K
- Similarity threshold
- Metadata filtering

## 4. Check Prompt Construction

Inspect the exact prompt sent to the model.

Verify:

- Correct context was included
- Context was not truncated
- Context is clearly separated
- Instructions are unambiguous

## 5. Check Generation

Compare the generated answer with the retrieved context.

Look for:

- Hallucination
- Unsupported claims
- Ignored context
- Incorrect reasoning

## 6. Isolate Components

Test independently:

1. Retrieval
2. Prompt construction
3. Generation

Find the earliest component where the error appears.

## 7. Apply the Smallest Fix

Change only the component responsible for the problem.

Possible fixes:

- Chunking
- Retrieval threshold
- Top-K
- Prompt wording
- Context formatting
- Generation configuration

## 8. Reproduce

Run the same failed query after the fix.

Confirm that the behavior improves.

## 9. Run Regression Evaluation

Run the complete Day 29 evaluation suite.

Make sure existing correct answers do not become incorrect.

## 10. Document

Record:

- Failure
- Root cause
- Trace
- Fix
- Before score
- After score
- Regression result
"""


with open(

    "AI_DEBUGGING_RUNBOOK.md",

    "w"

) as file:

    file.write(
        runbook
    )


print(
    "✅ AI_DEBUGGING_RUNBOOK.md saved"
)


# ================================================================
# 30. FINAL PROJECT SUMMARY
# ================================================================

print("\n\n")

print("=" * 100)

print(
    "🎯 DAY 33 — AI SYSTEM DEBUGGING COMPLETE"
)

print("=" * 100)


print("""
IMPLEMENTED:

✅ Local Hugging Face LLM
✅ NO OpenAI API
✅ Local embeddings
✅ FAISS vector retrieval
✅ LangSmith configuration
✅ Structured JSON logging
✅ Timestamp logging
✅ Session ID logging
✅ Step name logging
✅ Input summary logging
✅ Output summary logging
✅ Latency logging
✅ Retrieval isolation
✅ Prompt isolation
✅ Generation isolation
✅ Three reproducible failure cases
✅ Failure analysis
✅ Improved grounding prompt
✅ Improved retrieval configuration
✅ Two fixes tested
✅ Before/after evaluation
✅ Debugging report
✅ Debugging runbook

FILES CREATED:

📄 debugging_logs.jsonl
📄 before_after_evaluation.json
📄 day30_debugging_report.json
📄 AI_DEBUGGING_RUNBOOK.md

==============================================================

FOR YOUR ACTUAL DAY 33 SUBMISSION:

1. Replace the demonstration documents with your Day 32
   knowledge base/vector store.

2. Replace F1, F2 and F3 with the three REAL Day 32
   failures whose correctness or groundedness score was < 2.

3. Use the actual Day 32 evaluation scores.

4. Open LangSmith and inspect the three traces.

5. Record the actual root cause:
      Retrieval
      Prompt
      Generation

6. Apply two fixes based on the actual root causes.

7. Run the complete Day 29 evaluation again.

8. Confirm that the fixes improve the two failures without
   causing regressions elsewhere.

==============================================================
""")


# ================================================================
# 31. DOWNLOAD/VIEW FILES
# ================================================================

print("\nGenerated files:")

print("📄 debugging_logs.jsonl")
print("📄 before_after_evaluation.json")
print("📄 day33_debugging_report.json")
print("📄 AI_DEBUGGING_RUNBOOK.md")

print("\n🎉 Notebook execution finished!")

✅ All libraries imported
⚠️ LANGSMITH_API_KEY not found
Running locally
OpenAI API: NOT USED
✅ Structured JSON logging enabled
📁 debugging_logs.jsonl created
✅ Knowledge base: 6 documents

Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✅ Embedding model loaded
✅ FAISS vector store created
✅ Retriever created

Loading local FLAN-T5-small...


Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] T5ForConditionalGeneration LOAD REPORT from: google/flan-t5-small
Key                         | Status  | 
----------------------------+---------+-
decoder.embed_tokens.weight | MISSING | 
encoder.embed_tokens.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...



----------------------------------------------------------------------------------------------------
STEP 3 — GENERATION
----------------------------------------------------------------------------------------------------
_____))   is is is is is is is is is is  )))))))  in  is a a                            and    with withsspal  in in in in in --a-aaa-   with with withpals pal--   in in insssxx in-ssxx--s




FAILURE ANALYSIS — F1

Query:
What is LangSmith used for?

Expected:
LangSmith is used for observing, evaluating and debugging applications built with language models.

Actual:
_____) is is is is   sssiii in in  ))schiachii....................................................................................................................................

Retrieved evidence:

Document 1:
LangSmith is a platform for observing, evaluating and
        debugging applications built with language models.
        It provides traces that allow developers to inspect
        individual e

{"timestamp": "2026-09-11T17:26:23.220048+00:00", "session_id": "4f0bdf61-1eef-4f15-a27e-2e18c56f2162", "step_name": "fixed_generation", "input_summary": "\nYou are a factual and grounded question-answering assistant.\n\nUse ONLY the information explicitly provided in the context.\n\nRules:\n\n1. Do not use outside knowledge.\n2. Do not invent facts.\n3. Do not make unsupported claims.\n4. If the answer is not contained in the context,\n   respond exactly:\n   I don't have enough information in the context.\n5. Keep the answer concise.\n6. Every factual statement must be supported by the context.\n\nContext:\nDocument 1:\n\n        LangSmith is a platform for observing, evaluating and\n        debugging applications built with language models.\n        It provides traces that allow developers to inspect\n        individual execution steps.\n        \n\nDocument 2:\n\n        LangChain is a framework for developing applications\n        powered by language models. It provides components


Expected:
LangSmith is used for observing, evaluating and debugging applications built with language models.

New answer:
_____) is is of is)s)ssss)



####################################################################################################
TESTING FIX — F2
####################################################################################################


{"timestamp": "2026-09-11T17:26:33.600059+00:00", "session_id": "c10ce419-05af-417a-aa99-a757c5867dd0", "step_name": "fixed_generation", "input_summary": "\nYou are a factual and grounded question-answering assistant.\n\nUse ONLY the information explicitly provided in the context.\n\nRules:\n\n1. Do not use outside knowledge.\n2. Do not invent facts.\n3. Do not make unsupported claims.\n4. If the answer is not contained in the context,\n   respond exactly:\n   I don't have enough information in the context.\n5. Keep the answer concise.\n6. Every factual statement must be supported by the context.\n\nContext:\nDocument 1:\n\n        Retrieval-Augmented Generation, commonly called RAG,\n        combines information retrieval with language generation.\n        A retriever searches a knowledge base for relevant documents,\n        and the language model uses those documents as context.\n        \n\nDocument 2:\n\n        Grounded generation means that an AI model should produce\n        an


Expected:
RAG combines information retrieval with language generation.

New answer:
_____))) of of isa))))))))))))))))))))))))))))))))))))))))))))))))))))))))))))))))))))))))))))))))))))))))))))))))))))))))))))))))))))))))))))))))))))))))))


BEFORE vs AFTER EVALUATION


F1
Before: 0
After: 0
Improved: False


F2
Before: 0
After: 0
Improved: False


F3
Before: 0
After: 0
Improved: False

✅ before_after_evaluation.json saved
✅ day30_debugging_report.json saved
✅ AI_DEBUGGING_RUNBOOK.md saved



🎯 DAY 33 — AI SYSTEM DEBUGGING COMPLETE

IMPLEMENTED:

✅ Local Hugging Face LLM
✅ NO OpenAI API
✅ Local embeddings
✅ FAISS vector retrieval
✅ LangSmith configuration
✅ Structured JSON logging
✅ Timestamp logging
✅ Session ID logging
✅ Step name logging
✅ Input summary logging
✅ Output summary logging
✅ Latency logging
✅ Retrieval isolation
✅ Prompt isolation
✅ Generation isolation
✅ Three reproducible failure cases
✅ Failure analysis
✅ Improved grounding prompt
✅ Improved retrieval configuration